# Homework 7: Analyzing LFP

This homework will use an EEG dataset from Grenoble, France (https://zenodo.org/records/2358753). In this dataset, subjects were instrumented with 16-channel EEGs and asked to close and then open their eyes. The eyes-closed state is associated with low-frequency alpha oscillations, and in this assignment you will look at those oscillations in different ways.

## How to complete and submit this homework (read this first!)

This works the same in Google Colab, VS Code, or Jupyter.

1. **Put your NetID** in the setup cell below.
2. **Each problem ends with two `ANSWER` cells**: a code cell whose first line is `# ANSWER 1B`, and a text cell that starts with **ANSWER 1B: written answer**. **Everything from the code cell through the written-answer cell is graded together.** Add as many cells as you like between them (more code, more figures, notes). Work anywhere else in the notebook is treated as scratch and isn't graded.
3. **Write your answers in full sentences** in the written-answer cell, and refer to your figures ("the left panel shows..."). Explain your reasoning. Many questions have more than one good approach, and an approach different from ours is fine if you justify it.
4. **Do not delete the `ANSWER` cells or change their first lines.**
5. **If you can't finish a problem, make sure its cells still run without an error** (comment out the broken lines). An error stops "Run All", and every problem after it will then have no output.
6. **Before submitting:** restart and run everything from the top, then check that the last cell in the notebook printed its message.
   - Colab: *Runtime &rarr; Restart session and run all*
   - VS Code / Jupyter: *Restart* the kernel, then *Run All*
7. **Submit the `.ipynb` file to Canvas, not a PDF.** Name it `hw7_<netid>.ipynb`. Your figures are saved inside the file.
   - Colab: *File &rarr; Download &rarr; Download .ipynb*
   - VS Code / Jupyter: save (Ctrl/Cmd-S), then upload the `.ipynb` file from your computer.

In [ ]:
NETID = ""   # <-- your Rice NetID, e.g. "abc12"

import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.signal as signal

# Download the data if it isn't next to this notebook (e.g., on Colab)
DATA_URL = 'https://raw.githubusercontent.com/ckemere/elec380/main/HW7/'
for fname in ['subject_06.npz', 'subject_08.npz', 'subject_12.npz']:
    if not os.path.exists(fname):
        print('Downloading', fname)
        urllib.request.urlretrieve(DATA_URL + fname, fname)

print('numpy', np.__version__, '| scipy', scipy.__version__)
assert tuple(int(x) for x in scipy.__version__.split('.')[:2]) >= (1, 12), \
    'This homework needs scipy 1.12 or newer (for ShortTimeFFT). Try: pip install --upgrade scipy'

def load_subject(subject_number):
    # Returns eeg_data (samples x 16 channels), fs, eyes-closed and eyes-opened sample indices, channel labels
    d = np.load('subject_{:02d}.npz'.format(subject_number))
    return (d['eeg_data'], int(d['fs']), d['eyes_closed_indices'].ravel(),
            d['eyes_opened_indices'].ravel(), d['channel_labels'])

To begin with, we'll use data from **Subject 6**. Let's take a look at the data from two channels &mdash; channel 0 and channel 13. Channel 0 is the 'Fp1' channel (pre-frontal, over the left eye). Channel 13 is the 'O1' channel (occipital, on the back left of the skull).

In [ ]:
eeg_data, fs, eyes_closed_indices, eyes_opened_indices, channel_labels = load_subject(6)
t_eeg = np.arange(eeg_data.shape[0])/fs   # timestamps based on sampling rate (512 Hz)
t = t_eeg

for ch, lab in enumerate(channel_labels):
    print(ch, lab)

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(12,6))

ax[0].plot(t, eeg_data[:,0])  # Plot channel 0 (Fp1)
ax[1].plot(t, eeg_data[:,13]) # Plot channel 13 (O1)

# Plot the moments when the eyes closed and the eyes opened according to the experimenter
for a in ax:
    for closed in eyes_closed_indices:
        a.axvline(t[closed], color='r')
    for opened in eyes_opened_indices:
        a.axvline(t[opened], color='g')

We immediately notice a few things here.

First, there is a large artifact when the eyes are opened and closed. The artifact is larger in the channel near the eyes and smaller in the channel on the back of the head.

Second, the experimenter coding of when the eyes are closed and opened is slightly offset from when it happens in the experiment. We won't worry about this, but Problem 4 below (required for ELEC 587, extra credit for ELEC 380) asks you to correct for it.

What does the data look like if we zoom in around the eye closing period? Looking at the indices, it seems like the timing is better for the second event &mdash; let's use that one to plot.

In [ ]:
closed_index = eyes_closed_indices[1] # Index of second eyes closing

fig, ax = plt.subplots(2,1, figsize=(12,6), sharex=True)

ax[0].plot(t[closed_index - 1024: closed_index + 1024], eeg_data[closed_index - 1024: closed_index + 1024, 0])
ax[1].plot(t[closed_index - 1024: closed_index + 1024], eeg_data[closed_index - 1024: closed_index + 1024, 13])
ax[0].axvline(t[closed_index], color='r')
ax[1].axvline(t[closed_index], color='r')

# Problem 1: Looking at data in the frequency domain

Let's begin by asking a question. What channel has the highest change in power in the alpha band during the eyes-closed state? To answer this question, we will extract the signal during the eyes-closed and eyes-open periods. The experimenters specify that these periods last 10 s, but we've already seen that the onsets/offsets are pretty noisy (maybe 1&ndash;4 seconds after the actual event). So let's extract 4-second windows of signal starting 4 seconds after each cue.

In [ ]:
eyes_closed_signals = []
eyes_opened_signals = []
window_indices = np.arange(4*fs) + 4*fs # 4 s of samples starting 4 s after the cue

for closed in eyes_closed_indices:
    eyes_closed_signals.append(eeg_data[closed + window_indices,:])

for opened in eyes_opened_indices:
    eyes_opened_signals.append(eeg_data[opened + window_indices,:])

Now, let's calculate the "Power Spectral Density" for each of these little windows for channel 0 and channel 13 (eyes closed in red, eyes open in green).

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(12,6), sharex=True, sharey=True)

for i in range(5):
    # Calculate and plot the PSDs for channel 0
    f, Pxx = signal.welch(eyes_closed_signals[i][:,0], fs, nperseg=256)
    ax[0].semilogy(f,Pxx, 'r')
    f, Pxx = signal.welch(eyes_opened_signals[i][:,0], fs, nperseg=256)
    ax[0].semilogy(f,Pxx, 'g')

    # Calculate and plot the PSDs for channel 13
    f, Pxx = signal.welch(eyes_closed_signals[i][:,13], fs, nperseg=256)
    ax[1].semilogy(f,Pxx, 'r')
    f, Pxx = signal.welch(eyes_opened_signals[i][:,13], fs, nperseg=256)
    ax[1].semilogy(f,Pxx, 'g')
ax[0].set_ylabel('Power (Channel 0)')
ax[1].set_ylabel('Power (Channel 13)')
ax[1].set_xlabel('Frequency (Hz)')

To answer our question, we have to decide what frequency band alpha should be. Let's zoom in on our spectra in the relevant range.

In [ ]:
fig, ax = plt.subplots(16,2, figsize=(8,45), sharex=True, sharey=True)

nperseg = 256

for ch in range(16):
    eyes_closed_Pxx = []
    eyes_opened_Pxx = []
    for i in range(5):
        f, Pxx = signal.welch(eyes_closed_signals[i][:,ch], fs, nperseg=nperseg)
        ax[ch,0].plot(f,Pxx, 'r.-')
        eyes_closed_Pxx.append(Pxx)
        f, Pxx = signal.welch(eyes_opened_signals[i][:,ch], fs, nperseg=nperseg)
        ax[ch,0].plot(f,Pxx, 'g.-')
        eyes_opened_Pxx.append(Pxx)

    ax[ch,1].semilogy(f, np.mean(eyes_closed_Pxx, axis=0), 'r', label='Eyes Closed')
    ax[ch,1].semilogy(f, np.mean(eyes_opened_Pxx, axis=0), 'g', label='Eyes Opened')
    ax[ch,0].set_ylabel('Power (Ch {} {})'.format(ch, channel_labels[ch]))

ax[0,0].set_title('Individual trials')
ax[0,1].set_title('Average of 5 trials')
ax[0,0].set_xlim(2,16)
ax[0,1].legend()

## Problem 1A (10 pts): What frequency is the alpha rhythm for this subject?

Recall that naively using the Fourier transform on the data will result in frequency estimates that may be noisy. The Welch PSD method averages the results of smaller, overlapping windows (with size set by the `nperseg` parameter). This results in less noisy estimates, but lower frequency resolution (shorter window &harr; lower frequency resolution). Try several values of `nperseg` and see whether that changes your answer.

Make at least one figure that shows the evidence for your answer.

In [ ]:
# ANSWER 1A
# your code here

**ANSWER 1A: written answer**

- What is the alpha frequency for Subject 6? Which channel(s) did you base this on?
- What values of `nperseg` did you try, and did they change your estimate? Why or why not?

_Your answer:_

## Problem 1B (20 pts): Which channel shows the largest alpha signal when the eyes are closed? Which shows the largest difference between eyes open and eyes closed?

Below is a function that combines the cells above to calculate the average alpha power for a given `frequency` (the alpha frequency you decided on) and `eeg_channel`. Use it (or modify it) to compute, for every channel, the eyes-closed alpha power and the ratio of eyes-closed to eyes-open alpha power. The answer cell has a loop to fill in and an example of how to plot the result.

In [ ]:
def extract_alpha_power(eeg_channel=0, frequency=10, nperseg=256, window_size=4.0, window_offset=4.0):
    eyes_closed_signals = []
    eyes_opened_signals = []
    window_indices = np.arange(int(window_size*fs)) + int(window_offset*fs) # index into the data for the time windows

    for closed in eyes_closed_indices:
        eyes_closed_signals.append(eeg_data[closed + window_indices,:])

    for opened in eyes_opened_indices:
        eyes_opened_signals.append(eeg_data[opened + window_indices,:])

    # axis 0 is trials, axis 1 is time, axis 2 is channel
    f, eyes_closed_Pxx = signal.welch(np.array(eyes_closed_signals), fs, axis=1, nperseg=nperseg)
    f, eyes_opened_Pxx = signal.welch(np.array(eyes_opened_signals), fs, axis=1, nperseg=nperseg)

    fidx = np.abs(f - frequency).argmin()
    print('Channel {}: using frequency {} ({} specified).'.format(eeg_channel, f[fidx], frequency))

    return np.mean(eyes_closed_Pxx[:,fidx,eeg_channel], axis=0), np.mean(eyes_opened_Pxx[:,fidx,eeg_channel], axis=0)


# Example data point
eyes_closed_power, eyes_opened_power = extract_alpha_power(eeg_channel=13, frequency=6.9)
print(eyes_closed_power)

In [ ]:
# ANSWER 1B
alpha_power = []
alpha_power_ratio = []
for ch in range(16):
    pass  # replace this line: call extract_alpha_power(), compute the ratio, and append to both lists

# Plotting code
if len(alpha_power) == 16 and len(alpha_power_ratio) == 16:
    fig, ax = plt.subplots(1,2, figsize=(18,4))
    plot_labels = ['{} ({})'.format(lab, ch) for ch, lab in enumerate(channel_labels)]
    ax[0].bar(plot_labels, alpha_power)
    ax[0].tick_params(axis='x', labelrotation=90)
    ax[0].set_title('Alpha Power (Eyes Closed)')
    ax[1].bar(plot_labels, alpha_power_ratio)
    ax[1].tick_params(axis='x', labelrotation=90)
    ax[1].set_title('Alpha Power Ratio (Closed / Open)')

**ANSWER 1B: written answer**

- Which channel has the largest eyes-closed alpha power? Which has the largest eyes-closed / eyes-open ratio?
- Where on the head are they? Does it make sense that these locations show the strongest alpha?

_Your answer:_

## Problem 1C (10 pts): Which channel has the most noise?

Which channel has the most line noise (which is at 50 Hz in France!)? You can use windows like the ones above, though line noise shouldn't be different with eyes open and eyes closed. Alternatively, look at the whole session. Choose an approach and make a figure that supports your answer.

In [ ]:
# ANSWER 1C
# your code here

**ANSWER 1C: written answer**

- Which channel has the most line noise, and how did you measure it?

_Your answer:_

# Problem 2: Calculating a Fourier-based spectrogram

The "Short Time Fourier Transform" (STFT) is the technical term for the standard Fourier-transform-based spectrogram. To avoid the issues that come from sharp window boundaries, the STFT uses windows with rounded edges. In the example code below, we use one of the standard windows, the "Hamming window". Other common choices include Gaussian, triangular, and Hann windows. As we discussed in class, the choice of the window length (`Nx` in the `get_window()` call below) directly affects time/frequency resolution &mdash; longer windows have better resolution in frequency and worse resolution in time. The `hop` parameter specifies how far the window slides each step. Making `hop` smaller costs more computation but can give better temporal resolution (keeping in mind that the window size also affects that!).

The code below calculates and plots the spectrogram for channel 0. The choice `Nx=2048` gives high resolution in frequency.

In [ ]:
hamm_win = signal.get_window('hamming', Nx=2048)
SFT = signal.ShortTimeFFT(hamm_win, hop=32, fs=fs, scale_to='psd') # 'psd' scaling affects how the signal is normalized across frequency
stft = SFT.stft(eeg_data, axis=0)

fig, ax = plt.subplots(1,1,figsize=(6,4))
im = ax.imshow((abs(stft[:,0,:])), origin='lower', aspect='auto',
                 extent=SFT.extent(eeg_data.shape[0]), cmap='magma', vmax=10)
fig.colorbar(im, label="Magnitude $|S_x(t, f)|$")
ax.set_ylabel('Frequency (Hz)')
ax.set_xlabel('Time (s)')
ax.set_ylim(0, 30)
[ax.axvline(t_eeg[idx], color='white', alpha=0.25) for idx in eyes_closed_indices]
[ax.axvline(t_eeg[idx], linestyle=':', color='white', alpha=0.25) for idx in eyes_opened_indices];

Different subjects (and channels) have very different signal amplitudes, so a fixed `vmax=10` doesn't work for everyone. The function below plots a spectrogram of one channel in a standard way: it removes the channel's mean (which otherwise creates large artifacts at the start and end), and sets the color scale from the 99th percentile of the 2&ndash;30 Hz power. You are welcome to use it in Problems 2A and 2B.

In [ ]:
def plot_spectrogram(ax, x, fs, closed_indices, opened_indices, title, Nx=2048, hop=32):
    # x is a single channel (1-D array). Returns the image so you can add a colorbar.
    x = x - np.mean(x)
    SFT = signal.ShortTimeFFT(signal.get_window('hamming', Nx=Nx), hop=hop, fs=fs, scale_to='psd')
    S = np.abs(SFT.stft(x))
    vmax = np.percentile(S[(SFT.f > 2) & (SFT.f < 30)], 99)
    im = ax.imshow(S, origin='lower', aspect='auto', extent=SFT.extent(len(x)), cmap='magma', vmax=vmax)
    ax.set_ylim(0, 30)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)
    for idx in closed_indices:
        ax.axvline(idx/fs, color='white', alpha=0.4)
    for idx in opened_indices:
        ax.axvline(idx/fs, color='white', linestyle=':', alpha=0.4)
    return im

## Problem 2A (10 pts): Spectrogram of alpha

Using the channel you settled on as "best" in Problem 1, plot the spectrogram of Subject 6 through the eye closings and openings. Try other values of `Nx` to see how they change the spectrogram. Show whichever version(s) you think best support your answer.

In [ ]:
# ANSWER 2A
# your code here

**ANSWER 2A: written answer**

- What happens in the alpha band when the eyes close, and when they open?
- How did changing `Nx` affect what you could see?

_Your answer:_

## Problem 2B (15 pts): Does the frequency of alpha sweep?

In Subject 6, it seems like the frequency of the alpha rhythm sweeps downward slightly following eye closing. Load and plot spectrograms from Subjects 8 and 12. Do you see the same phenomenon in these subjects?

Use `load_subject(8)` and `load_subject(12)`, and **load them into new variable names** (e.g., `eeg_8, fs_8, closed_8, opened_8, labels_8 = load_subject(8)`) so that `eeg_data` still holds Subject 6 for Problem 3. Alpha is weaker in these subjects and the best channel may be different.

In [ ]:
# ANSWER 2B
# your code here

**ANSWER 2B: written answer**

- Which channel(s) did you use for each subject, and why?
- Do you see a downward sweep after eye closing in Subject 8? In Subject 12? How confident are you?

_Your answer:_

# Problem 3: Linear filters

When we look at our power spectra, it's obvious that there is lots of energy at frequencies other than the alpha band. Let's design a linear filter to isolate the alpha signal in our EEG data. The `firls()` function designs a filter using "least squares". The parameters describe what we want from it in terms of frequency: `bands=[0, 5, 7, 12, 14, 256]` with `desired=[0, 0, 1, 1, 0, 0]` says that we want no signal at frequencies below 5 Hz, full signal between 7 and 12 Hz, and no signal above 14 Hz. The `numtaps` parameter is how long the filter is. For this kind of filter the number should be odd. We initially choose 33, which is about 0.06 s (roughly half a cycle of an 8 Hz signal at fs = 512 Hz), but you will explore the effects of longer filters.

In [ ]:
b = signal.firls(numtaps=33, bands=[0, 5, 7, 12, 14, 256], desired=[0, 0, 1, 1, 0, 0], fs=fs)

We can visualize how well a filter will work using the `freqz()` function.

In [ ]:
f, response = signal.freqz(b, fs=fs) # Filter response is a complex number. We can ignore the phase for now.

fig, ax = plt.subplots(1,2, sharey=True)
ax[0].plot(f, 20*np.log10(np.abs(response))) # 20*log10() gives the response in "dB"
ax[1].plot(f, 20*np.log10(np.abs(response)))
ax[1].set_xlim(0,40)
ax[0].set_ylabel('Response of filter (dB)')

The filter we've designed blocks frequencies pretty well above ~20 Hz, but it doesn't seem to be doing a great job at lower frequencies. Let's make it a bit longer &mdash; `numtaps=65` instead of 33.

In [ ]:
b2 = signal.firls(numtaps=65, bands=[0, 5, 7, 12, 14, 256], desired=[0, 0, 1, 1, 0, 0], fs=fs)
f, response2 = signal.freqz(b2, fs=fs)

fig, ax = plt.subplots(1,2, sharey=True)
ax[0].plot(f, 20*np.log10(np.abs(response)), label='numtaps=33')
ax[0].plot(f, 20*np.log10(np.abs(response2)), label='numtaps=65')
ax[1].plot(f, 20*np.log10(np.abs(response)))
ax[1].plot(f, 20*np.log10(np.abs(response2)))
ax[1].set_xlim(0,40)
ax[0].set_ylabel('Response of filter (dB)')
ax[0].legend()

Our new filter is attenuating both high and low frequencies. Now let's apply it to our EEG data. In general, when you are filtering, you should use the function `filtfilt()`. This slides the filter along the signal forward in time _and_ backward in time. The result is that signals will be smeared in time symmetrically forward and backward, as we are going to explore.

In [ ]:
alpha_data = signal.filtfilt(b2, 1, eeg_data, axis=0)

Now let's look at the result in the time domain. We'll plot the same segment of signal on channels 0 and 13 as we did above, but this time also the alpha-band signal.

In [ ]:
closed_index = eyes_closed_indices[1] # Index of second eyes closing

fig, ax = plt.subplots(2,2, figsize=(12,6), sharex=True)

ax[0,0].plot(t[closed_index - 1024: closed_index + 1024], eeg_data[closed_index - 1024: closed_index + 1024, 0])
ax[1,0].plot(t[closed_index - 1024: closed_index + 1024], eeg_data[closed_index - 1024: closed_index + 1024, 13])

ax[0,1].plot(t[closed_index - 1024: closed_index + 1024], alpha_data[closed_index - 1024: closed_index + 1024, 0], color='coral')
ax[1,1].plot(t[closed_index - 1024: closed_index + 1024], alpha_data[closed_index - 1024: closed_index + 1024, 13], color='coral')

[a.axvline(t[closed_index], color='r') for a in ax.flatten()]

ax[0,0].set_title('Raw EEG Signal')
ax[0,1].set_title('Alpha Band Signal')
ax[0,0].set_ylabel('Channel 0 Signal')
ax[1,0].set_ylabel('Channel 13 Signal')

What about the spectrograms? Let's see how filtering changes them.

In [ ]:
hamm_win = signal.get_window('hamming', Nx=2048)
SFT = signal.ShortTimeFFT(hamm_win, hop=32, fs=fs, scale_to='psd')
stft = SFT.stft(eeg_data, axis=0)

fig, axs = plt.subplots(1,2,figsize=(12,4), sharey=True, sharex=True)
im = axs[0].imshow((abs(stft[:,0,:])), origin='lower', aspect='auto',
                 extent=SFT.extent(eeg_data.shape[0]), cmap='magma', vmax=10)
fig.colorbar(im, label="Magnitude $|S_x(t, f)|$")
axs[0].set_ylabel('Frequency (Hz)')
axs[0].set_ylim(0, 30)

stft2 = SFT.stft(alpha_data, axis=0)
im2 = axs[1].imshow((abs(stft2[:,0,:])), origin='lower', aspect='auto',
                 extent=SFT.extent(eeg_data.shape[0]), cmap='magma', vmax=10)
fig.colorbar(im2, label="Magnitude $|S_x(t, f)|$")

for ax in axs:
    [ax.axvline(t_eeg[idx], color='white', alpha=0.25) for idx in eyes_closed_indices]
    [ax.axvline(t_eeg[idx], linestyle=':', color='white', alpha=0.25) for idx in eyes_opened_indices]

It looks like most of the non-alpha signal is gone, but at the lowest frequencies there is still power. Looking back at the time-domain signal, we see that around time 38.0 s, channel 0 is quite far from 0 (unlike channel 13). Frequency 0 in a power spectrum/spectrogram is equal to the mean of the signal, so when it's nonzero, the mean is still far from zero.

### What about temporal smearing?

Let's construct a fake signal and filter it to highlight temporal smearing: a 1 s long pulse of 10 Hz "alpha" in the middle of 5 s of absolute silence.

In [ ]:
fake_t = np.arange(5*fs)/fs
fake_alpha = np.zeros(fake_t.shape[0])
fake_alpha[2*fs:3*fs] = np.sin(2*np.pi*10*fake_t[2*fs:3*fs])
plt.plot(fake_t, fake_alpha)

## Problem 3A (20 pts): Filter smearing

Filter the fake alpha signal with the two filters we created above (`b` and `b2`) using `filtfilt()`, and plot the results along with the original.

In [ ]:
# ANSWER 3A
fig, ax = plt.subplots(1,1, figsize=(12,3))
ax.plot(fake_t, fake_alpha, label='original')

# your code here: filter fake_alpha with b and b2 and plot both

ax.legend()

**ANSWER 3A: written answer**

- How far does each filtered signal extend beyond the original pulse, in cycles of the 10 Hz signal?
- Do you notice any distortion in the shape of the output signal? Where?

_Your answer:_

## Problem 3B (15 pts): Filter design and smearing

Now design a new filter with `numtaps=513` (same `bands` and `desired`), and look at both its frequency response and what it does to the fake alpha pulse.

In [ ]:
# ANSWER 3B
# your code here

**ANSWER 3B: written answer**

- How far does the 513-tap output extend beyond the pulse? Where do you notice the most signal distortion?
- What do you gain and what do you lose by using the longer filter?

_Your answer:_

# Problem 4 (30 pts; required for ELEC 587, extra credit for ELEC 380): Quantifying the alpha sweep

Using the Fp channels, re-align the eye-closing epochs to start when the eye-muscle artifact actually indicates the eyes closed. Extract each eye-closing event using the alpha-filtered data and calculate the average alpha spectrogram following eye closing. Extract the peak frequency at each time point. If you combine the 5 observations from Subjects 6, 8, and 12, for each time point you should have 15 samples. Fit a linear regression to these samples and report the statistical significance of its slope.

ELEC 380 students not attempting this problem: leave these cells as they are.

In [ ]:
# ANSWER 4
# your code here

**ANSWER 4: written answer**

- How did you find the true eye-closing times, and what window after eye closing did you analyze?
- What is the slope of the regression, and is it statistically significant? What does that tell you?

_Your answer:_

# Wavelet spectrogram (for edification, not graded)

Below is some sample code using the `ghostipy` package from Rice. You'll need to `pip install ghostipy` to use it (`!pip install ghostipy` in Colab). One thing the wavelet spectrogram makes clear (also apparent in the filtered time series) is that our alpha actually occurs in pulses, as opposed to as a continuous oscillation. This cell is skipped automatically if `ghostipy` is not installed.

In [ ]:
try:
    import ghostipy as gsp
except ImportError:
    gsp = None
    print('ghostipy not installed; skipping the wavelet example.')

if gsp is not None:
    gamma = 3
    beta = 25 # decrease beta to improve temporal resolution, increase to increase frequency resolution

    cwtcoefs, _, freq, cwtts, _ = gsp.cwt(
        eeg_data[:,13], freq_limits=[1, 30], fs=fs,
        wavelet=gsp.MorseWavelet(gamma=gamma, beta=beta))

    cwt_power = cwtcoefs.imag**2 + cwtcoefs.real**2
    ff = np.flipud(freq)

    # Wavelets sample across logarithmically spaced frequencies. To display these nicely:
    @plt.FuncFormatter
    def fake_log(x, pos):
        df = (np.log(ff[-1]) - np.log(ff[0]))/(len(ff) - 1)
        f = np.exp(df*x + np.log(ff[0]))
        return '{:4.0f}'.format(f)

    fig, ax = plt.subplots(1,1, figsize=(6,4))
    im = ax.imshow(cwt_power, interpolation=None, aspect='auto',
                   extent=[t_eeg[0], t_eeg[-1],0,len(freq)-1],
                   cmap=plt.cm.magma, vmax=250)
    ax.yaxis.set_major_formatter(fake_log)
    plt.colorbar(im)
    [ax.axvline(t_eeg[idx], color='white', alpha=0.25) for idx in eyes_closed_indices]
    [ax.axvline(t_eeg[idx], linestyle=':', color='white', alpha=0.25) for idx in eyes_opened_indices];

# End of homework

After *Restart and Run All*, the cell below should print its message. If it didn't, an error stopped the notebook somewhere above; find it and fix it (or comment it out) before submitting.

In [ ]:
print('Reached the end of HW7.', 'NetID:', NETID if NETID else '(NETID is empty! Set it in the setup cell.)')